In [1]:
import sys
sys.path.insert(0, '/opt/PatchSorter/prototyping')

import psycopg2
import time
import math
import os
import io
import numpy as np
from utils import (
    DB_PARAMS,
    HierarchicalGridIndexIJ,
    HierarchicalGridIndexZOrder,
    HierarchicalGridSQLRegistryIJ,
    HierarchicalGridSQLRegistryZOrder,
)
from shapely.geometry import box as shapely_box

DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = False
print("Connected to DB:", DB_PARAMS['database'])
import table_seeding
from table_seeding import CELL_SIZE, LEVEL, COORD_RANGE

TOTAL_ROWS = 100_000_000

Connected to DB: testdb


In [2]:
scaled_cell_size = CELL_SIZE / (2 ** LEVEL)
grid_width = int(COORD_RANGE / scaled_cell_size)
grid_height = int(COORD_RANGE / scaled_cell_size)

print(f"Grid dimensions at level {LEVEL}: {grid_width} x {grid_height}")

Grid dimensions at level 11: 2048 x 2048


## 1 – Create the bench table

In [ ]:
table_seeding.setup_schema(True)

## 2 – Populate bench_patches directly inside Postgres (parallel generate_series)

Data is generated entirely server-side using `generate_series` + the SQL helper
functions, split across multiple parallel worker connections so Postgres can
saturate available CPU cores.  No Python data generation, no network transfer
of row data, no text parsing — just binary server-side inserts.

In [ ]:
table_seeding.populate(total_rows=TOTAL_ROWS, num_workers=8)

## 3 – Create B-tree indices

In [ ]:

table_seeding.create_indexes()


## 4 – Pick representative cell IDs for benchmarking

We pick a point near the centre of the coordinate space and derive cell IDs at the chosen resolution level.

In [3]:
ij_index = HierarchicalGridIndexIJ(cell_size=CELL_SIZE)
z_index  = HierarchicalGridIndexZOrder(cell_size=CELL_SIZE)

sample_x, sample_y = COORD_RANGE / 2.0, COORD_RANGE / 2.0

sample_cell_ij = ij_index.point_to_cell(sample_x, sample_y, LEVEL)
sample_cell_z  = z_index .point_to_cell(sample_x, sample_y, LEVEL)

# Raw i, j coordinates (same formula as the INSERT above)
scaled_size  = CELL_SIZE / (2 ** LEVEL)
sample_i     = int(math.floor(sample_x / scaled_size)) & 0x1FFFFFFF
sample_j     = int(math.floor(sample_y / scaled_size)) & 0x1FFFFFFF

print(f"Sample point  : ({sample_x}, {sample_y})")
print(f"Cell IJ       : {sample_cell_ij}")
print(f"Cell Z-order  : {sample_cell_z}")
print(f"Raw i         : {sample_i}")
print(f"Raw j         : {sample_j}")

Sample point  : (0.5, 0.5)
Cell IJ       : 3170534687424644096
Cell Z-order  : 3170534137671974912
Raw i         : 1024
Raw j         : 1024


## 5 – Query 1: time to retrieve the *first* patch from a single grid cell

All four index strategies are exercised with a `LIMIT 1` point-lookup.

In [4]:
REPEATS = 5

def median(lst):
    s = sorted(lst)
    n = len(s)
    return s[n // 2] if n % 2 else (s[n//2 - 1] + s[n//2]) / 2

def time_query(cur, sql, params=None, repeats=REPEATS):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        cur.execute(sql, params)
        cur.fetchall()
        times.append(time.perf_counter() - t0)
    return times


In [ ]:

with conn.cursor() as cur:
    cur.execute("SET enable_seqscan = OFF;")

    sql_ij_q1 = """
        SELECT id, embed_x, embed_y
        FROM   bench_patches
        WHERE  grid_id_ij = %s
        LIMIT  1;
    """
    times_ij_q1 = time_query(cur, sql_ij_q1, (sample_cell_ij,))

    sql_z_q1 = """
        SELECT id, embed_x, embed_y
        FROM   bench_patches
        WHERE  grid_id_z = %s
        LIMIT  1;
    """
    times_z_q1 = time_query(cur, sql_z_q1, (sample_cell_z,))

    sql_i_idx_q1 = """
        SELECT id, embed_x, embed_y
        FROM   bench_patches
        WHERE  grid_i_indexed = %s AND grid_j_indexed = %s
        LIMIT  1;
    """
    times_i_idx_q1 = time_query(cur, sql_i_idx_q1, (sample_i, sample_j))

    cur.execute("SET enable_seqscan = ON;")

    # Unindexed: allow seq scan (no index to force off)
    sql_i_unidx_q1 = """
        SELECT id, embed_x, embed_y
        FROM   bench_patches
        WHERE  grid_i_unindexed = %s AND grid_j_unindexed = %s
        LIMIT  1;
    """
    times_i_unidx_q1 = time_query(cur, sql_i_unidx_q1, (sample_i, sample_j))

print("=== Query 1: LIMIT 1 point-lookup ===")
print(f"  IJ composite    : {[f'{t:.4f}' for t in times_ij_q1]}  → median {median(times_ij_q1)*1000:.2f} ms")
print(f"  Z-order         : {[f'{t:.4f}' for t in times_z_q1]}  → median {median(times_z_q1)*1000:.2f} ms")
print(f"  i+j indexed     : {[f'{t:.4f}' for t in times_i_idx_q1]}  → median {median(times_i_idx_q1)*1000:.2f} ms")
print(f"  i+j unindexed   : {[f'{t:.4f}' for t in times_i_unidx_q1]}  → median {median(times_i_unidx_q1)*1000:.2f} ms")

## 6 – Query 2: retrieve *all* patches in a square spatial box

| Strategy | Approach | Notes |
|---|---|---|
| Z-order | `BETWEEN z_min AND z_max` | Single range scan; minor false positives at edges |
| IJ composite | `= ANY(cell_list)` | Exact; one probe per cell |
| i+j indexed | `i BETWEEN i_min AND i_max AND j BETWEEN j_min AND j_max` | Two-column range; index on i filters rows, j checked as recheck |
| i+j unindexed | same predicate, full seq scan | Baseline: no index at all |

In [ ]:
BOX_HALF = COORD_RANGE * 0.02   # ±5 % → 10 % side length

qbox = shapely_box(
    sample_x - BOX_HALF, sample_y - BOX_HALF,
    sample_x + BOX_HALF, sample_y + BOX_HALF,
)

# Z-order: use polygon_to_morton_ranges for tight, merged BETWEEN ranges
morton_ranges = z_index.polygon_to_morton_ranges(qbox, LEVEL, max_recursion_depth=7)
print(f"Z-order: {len(morton_ranges)} Morton range(s) covering the box:")
for lo, hi in morton_ranges:
    print(f"  BETWEEN {lo} AND {hi}  (span {hi - lo + 1:,})")

# IJ composite
ij_cells = [sample_cell_ij]
print(f"IJ:      {len(ij_cells):,} cells in box  →  = ANY(array of {len(ij_cells):,} values)")

# Raw i/j range (from bounding box corners)
i_min = int(math.floor((sample_x - BOX_HALF) / scaled_size)) & 0x1FFFFFFF
i_max = int(math.floor((sample_x + BOX_HALF) / scaled_size)) & 0x1FFFFFFF
j_min = int(math.floor((sample_y - BOX_HALF) / scaled_size)) & 0x1FFFFFFF
j_max = int(math.floor((sample_y + BOX_HALF) / scaled_size)) & 0x1FFFFFFF
print(f"i range: [{i_min}, {i_max}]  j range: [{j_min}, {j_max}]")
print(f"  → covers {(i_max - i_min + 1) * (j_max - j_min + 1):,} cells")

In [ ]:
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

# Create a figure and axis for the plot
fig, ax = plt.subplots(figsize=(10, 10))

# --- Draw hierarchical grid at LEVEL resolution ---
x_min_view = sample_x - BOX_HALF * 3
x_max_view = sample_x + BOX_HALF * 3
y_min_view = sample_y - BOX_HALF * 3
y_max_view = sample_y + BOX_HALF * 3

# Snap to the first grid line at or before the view extent
grid_x_start = math.floor(x_min_view / scaled_cell_size) * scaled_cell_size
grid_y_start = math.floor(y_min_view / scaled_cell_size) * scaled_cell_size

x = grid_x_start
while x <= x_max_view:
    ax.axvline(x, color='gray', linewidth=0.4, alpha=0.5)
    x += scaled_cell_size

y = grid_y_start
while y <= y_max_view:
    ax.axhline(y, color='gray', linewidth=0.4, alpha=0.5)
    y += scaled_cell_size

# Iterate through the Z-index ranges and plot rectangles
for lo, hi in morton_ranges:
    # Convert the lower and upper bounds of the range to points
    x_lo, y_lo = z_index.cell_to_point(lo)
    x_hi, y_hi = z_index.cell_to_point(hi)

    # Calculate the width and height of the rectangle
    width = x_hi - x_lo
    height = y_hi - y_lo

    # Add a rectangle to the plot
    rect = Rectangle((x_lo, y_lo), width, height, edgecolor='blue', facecolor='lightblue', alpha=0.3, linewidth=1)
    ax.add_patch(rect)

# Plot the query bounding box in red
qbox_rect = Rectangle(
    (sample_x - BOX_HALF, sample_y - BOX_HALF),
    BOX_HALF * 2, BOX_HALF * 2,
    edgecolor='red', facecolor='none', linewidth=2, label='Query box'
)
ax.add_patch(qbox_rect)

ax.set_xlim(x_min_view, x_max_view)
ax.set_ylim(y_min_view, y_max_view)
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_title(f"Hierarchical Grid (level {LEVEL}, cell size {scaled_cell_size:.4f}) + Z-Index Ranges")
ax.legend()
plt.show()

In [ ]:
# Build a multi-range Z-order query from polygon_to_morton_ranges
_z_conditions = " OR ".join(
    f"grid_id_z BETWEEN {lo} AND {hi}" for lo, hi in morton_ranges
)
sql_z_q2 = f"""
    SELECT COUNT(*)
    FROM   bench_patches
    WHERE  ({_z_conditions});
"""

with conn.cursor() as cur:
    cur.execute("SET enable_seqscan = OFF;")

    # --- Z-order: merged Morton ranges from polygon_to_morton_ranges ---
    times_z_q2 = time_query(cur, sql_z_q2)
    cur.execute(sql_z_q2)
    z_rows = cur.fetchone()[0]

    # --- IJ composite: exact cell enumeration via = ANY ---
    sql_ij_q2 = """
        SELECT COUNT(*)
        FROM   bench_patches
        WHERE  grid_id_ij = ANY(%s);
    """
    ij_cells_arr = list(ij_cells)
    times_ij_q2 = time_query(cur, sql_ij_q2, (ij_cells_arr,))
    cur.execute(sql_ij_q2, (ij_cells_arr,))
    ij_rows = cur.fetchone()[0]

    # --- i+j indexed: two-column BETWEEN ---
    sql_ij_idx_q2 = """
        SELECT COUNT(*)
        FROM   bench_patches
        WHERE  grid_i_indexed BETWEEN %s AND %s
          AND  grid_j_indexed BETWEEN %s AND %s;
    """
    times_ij_idx_q2 = time_query(cur, sql_ij_idx_q2, (i_min, i_max, j_min, j_max))
    cur.execute(sql_ij_idx_q2, (i_min, i_max, j_min, j_max))
    ij_idx_rows = cur.fetchone()[0]

    cur.execute("SET enable_seqscan = ON;")

print("=== Query 2: all patches in spatial box ===")
print(f"  Z-order  {len(morton_ranges)} range(s)       — rows={z_rows:,}        median {median(times_z_q2)*1000:.1f} ms   {[f'{t:.3f}s' for t in times_z_q2]}")
print(f"  IJ       = ANY(cells)    — rows={ij_rows:,}       median {median(times_ij_q2)*1000:.1f} ms   {[f'{t:.3f}s' for t in times_ij_q2]}")
print(f"  i+j idx  BETWEEN         — rows={ij_idx_rows:,}   median {median(times_ij_idx_q2)*1000:.1f} ms   {[f'{t:.3f}s' for t in times_ij_idx_q2]}")
print()
print(f"Note: Z-order uses {len(morton_ranges)} merged BETWEEN range(s) from polygon_to_morton_ranges — minimal false positives.")
print("      i+j indexed uses separate single-column indexes (bitmap AND).")

In [ ]:
with conn.cursor() as cur:
    cur.execute("SET enable_seqscan = OFF;")

    print("── Z-order multi-range BETWEEN ──────────────────────────────")
    cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + sql_z_q2)
    for row in cur.fetchall():
        print(row[0])

    print("\n── IJ = ANY ─────────────────────────────────────────────────")
    cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + sql_ij_q2, (ij_cells_arr,))
    for row in cur.fetchall():
        print(row[0])

    print("\n── i+j indexed BETWEEN ──────────────────────────────────────")
    cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + sql_ij_idx_q2,
                (i_min, i_max, j_min, j_max))
    for row in cur.fetchall():
        print(row[0])

    cur.execute("SET enable_seqscan = ON;")

    print("\n── i+j unindexed BETWEEN (seq scan baseline) ────────────────")
    cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + sql_ij_unidx_q2,
                (i_min, i_max, j_min, j_max))
    for row in cur.fetchall():
        print(row[0])

## 8 – Summary

| Query | Index | Approach | Notes |
|---|---|---|---|
| Point lookup (`LIMIT 1`) | IJ composite | `= <cell_id>` | Single B-tree probe on packed IJ key |
| Point lookup (`LIMIT 1`) | Z-order | `= <cell_id>` | Single B-tree probe on Morton key |
| Point lookup (`LIMIT 1`) | i+j indexed | `i=X AND j=Y` | Bitmap AND of two B-tree indexes |
| Point lookup (`LIMIT 1`) | i+j unindexed | `i=X AND j=Y` | Full sequential scan — baseline |
| Box scan (all rows) | Z-order | `BETWEEN z_min AND z_max` | One range scan; minor false positives at Z-curve edges |
| Box scan (all rows) | IJ composite | `= ANY(cell_list)` | One probe per cell; exact results |
| Box scan (all rows) | i+j indexed | `i BETWEEN … AND j BETWEEN …` | Bitmap AND; Postgres uses whichever index is more selective first |
| Box scan (all rows) | i+j unindexed | `i BETWEEN … AND j BETWEEN …` | Sequential scan — pure CPU/IO baseline |

**Key insight**: the Z-order index excels at bounding-box scans because Morton
encoding preserves 2-D locality in a single integer range.  Separate i/j
B-tree indexes require a bitmap AND of two range scans and incur significantly
more I/O for large boxes.  The unindexed columns provide a sequential-scan
baseline to measure the absolute cost floor.

## 9 – Query 3: Circular polygon — row-value bbox range + exact `grid_id_ij` residual filter

Two-step approach for an arbitrary polygon (here a circle):

1. **Python pre-filter** — enumerate every grid cell in the circle's bounding box,
   compute each cell-centre coordinate, and keep only those whose centre lies
   **inside** the circle.  The resulting list of `grid_id_ij` values is the exact
   cell set.
2. **SQL query** — two predicates in one pass:
   - `(grid_i_indexed, grid_j_indexed) >= (ci_min, cj_min) AND … <= (ci_max, cj_max)` —
     row-value comparison performs a single lexicographic range scan over the
     bbox, tightly bounding the rows the planner must inspect.
   - `AND grid_id_ij = ANY(%s)` — residual recheck on the packed IJ key that
     removes any remaining false positives (row-value ranges are lexicographic,
     not rectangular).

| Metric | Meaning |
|---|---|
| Bbox cells | Cells in the circle's bounding rectangle |
| Circle cells | Cells whose centre lies inside the circle (passed to SQL) |
| Excluded cells | Corner cells rejected by the Python pre-filter |
| Exclusion rate | Corners removed as % of bbox cells |

In [5]:
import shapely.geometry as sg

# ── Define the circular query polygon ────────────────────────────────────────
CIRCLE_RADIUS = COORD_RANGE * 0.02        # same scale as BOX_HALF in Query 2
circle_polygon = sg.Point(sample_x, sample_y).buffer(CIRCLE_RADIUS, resolution=64)

circ_xmin, circ_ymin, circ_xmax, circ_ymax = circle_polygon.bounds

# Grid i/j ranges covering the circle's bounding box
ci_min = int(math.floor(circ_xmin / scaled_size)) & 0x1FFFFFFF
ci_max = int(math.floor(circ_xmax / scaled_size)) & 0x1FFFFFFF
cj_min = int(math.floor(circ_ymin / scaled_size)) & 0x1FFFFFFF
cj_max = int(math.floor(circ_ymax / scaled_size)) & 0x1FFFFFFF

bbox_cells_ij = [
    (i, j)
    for i in range(ci_min, ci_max + 1)
    for j in range(cj_min, cj_max + 1)
]
print(f"Circle centre     : ({sample_x}, {sample_y}),  radius = {CIRCLE_RADIUS:.4f}")
print(f"Bounding box      : x=[{circ_xmin:.4f}, {circ_xmax:.4f}]  "
      f"y=[{circ_ymin:.4f}, {circ_ymax:.4f}]")
print(f"Grid i range      : [{ci_min}, {ci_max}]  ({ci_max - ci_min + 1} cells wide)")
print(f"Grid j range      : [{cj_min}, {cj_max}]  ({cj_max - cj_min + 1} cells tall)")
print(f"Bbox total cells  : {len(bbox_cells_ij):,}")

# ── Step 1: Python pre-filter — keep only (i, j) pairs inside the circle ─────
t_prefilter_start = time.perf_counter()

circle_ij_pairs = []    # (i, j) tuples for cells whose centre lies inside the circle
excluded_cells  = []    # (i, j) pairs rejected as bbox-corner false positives

for i, j in bbox_cells_ij:
    cx = (i + 0.5) * scaled_size   # cell-centre x
    cy = (j + 0.5) * scaled_size   # cell-centre y
    if circle_polygon.contains(sg.Point(cx, cy)):
        circle_ij_pairs.append((i, j))
    else:
        excluded_cells.append((i, j))

# Separate i and j lists for UNNEST — avoids psycopg2 record-type inference
circle_i_list = [i for i, j in circle_ij_pairs]
circle_j_list = [j for i, j in circle_ij_pairs]

prefilter_elapsed = time.perf_counter() - t_prefilter_start
excl_pct = len(excluded_cells) / len(bbox_cells_ij) * 100 if bbox_cells_ij else 0.0

print(f"\nStep 1 – Python pre-filter  [{prefilter_elapsed * 1000:.2f} ms]")
print(f"  Cells inside circle : {len(circle_ij_pairs):,}  (passed to SQL residual filter)")
print(f"  Cells excluded      : {len(excluded_cells):,}  ({excl_pct:.1f}% bbox corners removed)")

# ── Step 2: SQL — BETWEEN bbox scan + (i, j) IN UNNEST residual filter ───────
#
# The BETWEEN clauses drive the bitmap-AND B-tree index scan to narrow the
# candidate set to the bounding rectangle.
#
# The residual subquery unnests two parallel int[] arrays into (i, j) pairs
# and checks membership — no record type inference, no type mismatch.
sql_composite_range = """
    SELECT COUNT(*)
    FROM   bench_patches
    WHERE  grid_i_indexed BETWEEN %s AND %s
      AND  grid_j_indexed BETWEEN %s AND %s

"""

t_sql_start = time.perf_counter()
with conn.cursor() as cur:
    cur.execute("SET enable_seqscan = OFF;")
    times_circle = time_query(
        cur, sql_composite_range,
        (ci_min, ci_max, cj_min, cj_max), #, circle_i_list, circle_j_list),
    )
    cur.execute(sql_composite_range,
                (ci_min, ci_max, cj_min, cj_max)) #, circle_i_list, circle_j_list))
    circle_row_count = cur.fetchone()[0]
    cur.execute("SET enable_seqscan = ON;")
sql_elapsed = time.perf_counter() - t_sql_start

print(f"\nStep 2 – SQL  i BETWEEN + j BETWEEN  AND  (i,j) IN UNNEST(circle_i[], circle_j[])")
print(f"  i range           : [{ci_min}, {ci_max}]")
print(f"  j range           : [{cj_min}, {cj_max}]")
print(f"  UNNEST array size : {len(circle_ij_pairs):,} (i,j) pairs")
print(f"  Rows returned     : {circle_row_count:,}")
print(f"  Median query      : {median(times_circle) * 1000:.2f} ms  over {REPEATS} runs")
print(f"  All runs          : {[f'{t:.3f}s' for t in times_circle]}")
print(f"\n  Total elapsed (pre-filter + SQL) : "
      f"{(prefilter_elapsed + sql_elapsed) * 1000:.1f} ms")

Circle centre     : (0.5, 0.5),  radius = 0.0200
Bounding box      : x=[0.4800, 0.5200]  y=[0.4800, 0.5200]
Grid i range      : [983, 1064]  (82 cells wide)
Grid j range      : [983, 1064]  (82 cells tall)
Bbox total cells  : 6,724

Step 1 – Python pre-filter  [70.53 ms]
  Cells inside circle : 5,268  (passed to SQL residual filter)
  Cells excluded      : 1,456  (21.7% bbox corners removed)

Step 2 – SQL  i BETWEEN + j BETWEEN  AND  (i,j) IN UNNEST(circle_i[], circle_j[])
  i range           : [983, 1064]
  j range           : [983, 1064]
  UNNEST array size : 5,268 (i,j) pairs
  Rows returned     : 159,676
  Median query      : 28.72 ms  over 5 runs
  All runs          : ['0.039s', '0.034s', '0.029s', '0.028s', '0.028s']

  Total elapsed (pre-filter + SQL) : 258.1 ms


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle, Rectangle as MplRectangle

fig, ax = plt.subplots(figsize=(10, 10))

# ── Viewport ──────────────────────────────────────────────────────────────────
view_pad   = CIRCLE_RADIUS * 1.4
x_min_view = sample_x - view_pad
x_max_view = sample_x + view_pad
y_min_view = sample_y - view_pad
y_max_view = sample_y + view_pad

# ── Grid lines at LEVEL resolution ───────────────────────────────────────────
gx = math.floor(x_min_view / scaled_size) * scaled_size
while gx <= x_max_view:
    ax.axvline(gx, color='gray', linewidth=0.3, alpha=0.35)
    gx += scaled_size

gy = math.floor(y_min_view / scaled_size) * scaled_size
while gy <= y_max_view:
    ax.axhline(gy, color='gray', linewidth=0.3, alpha=0.35)
    gy += scaled_size

# ── Green: (i, j) pairs passed to the SQL residual filter ────────────────────
for i_val, j_val in circle_ij_pairs:
    rx, ry = i_val * scaled_size, j_val * scaled_size
    ax.add_patch(MplRectangle(
        (rx, ry), scaled_size, scaled_size,
        facecolor='lightgreen', edgecolor='green', alpha=0.55, linewidth=0.6,
    ))

# ── Orange: bbox corner cells excluded by the Python pre-filter ───────────────
for i_val, j_val in excluded_cells:
    rx, ry = i_val * scaled_size, j_val * scaled_size
    ax.add_patch(MplRectangle(
        (rx, ry), scaled_size, scaled_size,
        facecolor='lightsalmon', edgecolor='darkorange', alpha=0.55, linewidth=0.6,
    ))

# ── Circle and its bounding box ───────────────────────────────────────────────
ax.add_patch(Circle(
    (sample_x, sample_y), CIRCLE_RADIUS,
    edgecolor='red', facecolor='none', linewidth=2, zorder=5,
))
ax.add_patch(MplRectangle(
    (circ_xmin, circ_ymin),
    circ_xmax - circ_xmin, circ_ymax - circ_ymin,
    edgecolor='steelblue', facecolor='none', linewidth=1.5,
    linestyle='--', zorder=4,
))

ax.set_xlim(x_min_view, x_max_view)
ax.set_ylim(y_min_view, y_max_view)
ax.set_aspect('equal')
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_title(
    f"Circle query — BETWEEN bbox scan + (i,j) = ANY(circle_pairs)\n"
    f"radius = {CIRCLE_RADIUS:.4f},  cell size = {scaled_size:.4f},  level = {LEVEL}\n"
    f"Queried (green): {len(circle_ij_pairs):,} cells ({circle_row_count:,} rows)   "
    f"Excluded (orange): {len(excluded_cells):,} bbox corners ({excl_pct:.1f}%)"
)
legend_handles = [
    mpatches.Patch(facecolor='lightgreen',  edgecolor='green',
                   label=f'Passed to (i,j) = ANY(…) — {len(circle_ij_pairs):,} cells'),
    mpatches.Patch(facecolor='lightsalmon', edgecolor='darkorange',
                   label=f'Excluded by Python pre-filter — {len(excluded_cells):,} cells ({excl_pct:.1f}%)'),
    mpatches.Patch(facecolor='none',        edgecolor='red',
                   label='Query circle'),
    mpatches.Patch(facecolor='none',        edgecolor='steelblue',
                   label='Bounding box (BETWEEN range)'),
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()